In [1]:
import argparse
import logging
import pyspark.sql.functions as F
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import DoubleType, LongType, StringType, StructField, StructType, TimestampNTZType

In [2]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("taxi_data_processing") \
    .config("spark.sql.parquet.enableVectorizedReader", "false") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/20 14:37:01 WARN Utils: Your hostname, Adrians-M2-Macbook.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.97 instead (on interface en0)
26/03/20 14:37:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/20 14:37:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
input_green = "../data/parquet_spark/green/*/*/*"
input_yellow = "../data/parquet_spark/yellow/*/*/*"
output = "../data/parquet_spark/revenue/"

In [5]:

GREEN_SCHEMA = StructType([
    StructField("VendorID", LongType(), True),
    StructField("lpep_pickup_datetime", TimestampNTZType(), True),
    StructField("lpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("RatecodeID", DoubleType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("ehail_fee", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("payment_type", DoubleType(), True),
    StructField("trip_type", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
])

YELLOW_SCHEMA = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampNTZType(), True),
    StructField("tpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True),
])

In [6]:
def process_green_data(spark: SparkSession, input_green: str) -> DataFrame:
    logger.info(f"Processing green data from {input_green}")
    return spark.read.schema(GREEN_SCHEMA).parquet(input_green) \
        .withColumnsRenamed({
            "lpep_pickup_datetime": "pickup_datetime",
            "lpep_dropoff_datetime": "dropoff_datetime",
        }) \
        .withColumn("service_type", F.lit("green"))

In [7]:
def process_yellow_data(spark: SparkSession, input_yellow: str) -> DataFrame:
    logger.info(f"Processing yellow data from {input_yellow}")
    return spark.read.schema(YELLOW_SCHEMA).parquet(input_yellow) \
        .withColumnsRenamed({
            "tpep_pickup_datetime": "pickup_datetime",
            "tpep_dropoff_datetime": "dropoff_datetime",
        }) \
        .withColumn("service_type", F.lit("yellow"))

In [8]:
green_df = process_green_data(spark, input_green)
green_df.printSchema()
green_df.show(5)

INFO:__main__:Processing green data from ../data/parquet_spark/green/*/*/*
26/03/20 14:37:12 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/parquet_spark/green/*/*/*.
java.io.FileNotFoundException: File ../data/parquet_spark/green/*/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSo

root
 |-- VendorID: long (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: double (nullable = true)
 |-- trip_type: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- service_type: string (nullable = false)



+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------+
|VendorID|    pickup_datetime|   dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|service_type|
+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------+
|       2|2020-01-11 04:05:54|2020-01-11 04:13:49|                 N|       1.0|         129|         129|            1.0|         0.

In [9]:
yellow_df = process_yellow_data(spark, input_yellow)
yellow_df.printSchema()
yellow_df.show(5)

INFO:__main__:Processing yellow data from ../data/parquet_spark/yellow/*/*/*
26/03/20 14:39:48 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/parquet_spark/yellow/*/*/*.
java.io.FileNotFoundException: File ../data/parquet_spark/yellow/*/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDa

root
 |-- VendorID: long (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- service_type: string (nullable = false)

+--------+-------------------+-------------------+---------------+-------------+----------+----

In [10]:
def union_dataframes(green_df: DataFrame, yellow_df: DataFrame) -> DataFrame:
    common_columns = [column for column in green_df.columns if column in yellow_df.columns]
    return green_df.select(common_columns).union(yellow_df.select(common_columns))

In [11]:
def create_revenue_report(spark: SparkSession, df: DataFrame) -> DataFrame:
    df.createOrReplaceTempView("taxi_data")
    return spark.sql("""
        SELECT
            -- Revenue grouping
            PULocationID AS revenue_zone,
            date_trunc('month', pickup_datetime) AS revenue_month,
            service_type,

            -- Revenue calculation
            SUM(fare_amount) AS revenue_monthly_fare,
            SUM(extra) AS revenue_monthly_extra,
            SUM(mta_tax) AS revenue_monthly_mta_tax,
            SUM(tip_amount) AS revenue_monthly_tip_amount,
            SUM(tolls_amount) AS revenue_monthly_tolls_amount,
            SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
            SUM(total_amount) AS revenue_monthly_total_amount,
            SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

            -- Additional calculations
            AVG(passenger_count) AS avg_monthly_passenger_count,
            AVG(trip_distance) AS avg_monthly_trip_distance
        FROM taxi_data
        GROUP BY 1, 2, 3
    """)

In [12]:
def write_parquet(df: DataFrame, output: str) -> None:
    logger.info(f"Writing results to {output}")
    df.write.parquet(output, mode="overwrite")

In [16]:
df_unioned = union_dataframes(green_df, yellow_df)
df_unioned.groupBy("service_type", F.year("pickup_datetime").alias("year")).count().orderBy("service_type", "year").show()

+------------+----+--------+
|service_type|year|   count|
+------------+----+--------+
|       green|2008|       8|
|       green|2009|      46|
|       green|2010|       7|
|       green|2019|      19|
|       green|2020| 1734124|
|       green|2021| 1068726|
|       green|2041|       1|
|      yellow|2002|       2|
|      yellow|2003|      12|
|      yellow|2004|       1|
|      yellow|2008|     118|
|      yellow|2009|     293|
|      yellow|2011|       4|
|      yellow|2019|     131|
|      yellow|2020|24648828|
|      yellow|2021|30903958|
|      yellow|2022|      49|
|      yellow|2028|       1|
|      yellow|2029|       1|
|      yellow|2070|       1|
+------------+----+--------+
only showing top 20 rows


In [17]:
df_revenue = create_revenue_report(spark, df_unioned)
df_revenue.show(5)

+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|revenue_zone|      revenue_month|service_type|revenue_monthly_fare|revenue_monthly_extra|revenue_monthly_mta_tax|revenue_monthly_tip_amount|revenue_monthly_tolls_amount|revenue_monthly_improvement_surcharge|revenue_monthly_total_amount|revenue_monthly_congestion_surcharge|avg_monthly_passenger_count|avg_monthly_trip_distance|
+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|          38

In [18]:
write_parquet(df_revenue, output)

INFO:__main__:Writing results to ../data/parquet_spark/revenue/
